# Coordinates and affine composition

[Course index](../README.md) · [Week 10](../seminars/10_change_of_basis.md) · [Week 11](../seminars/11_affine_geometry.md)

**Predict → compute → explain → change an assumption.** Run top to bottom in a fresh Python kernel. GitHub previews do not execute widgets. Install the repository requirements once; no downloads occur in this notebook. All investigations have a paper route in the linked seminar sheets. A plot supports exploration, not a proof.

AI may help with the investigation if permitted. Write a prediction first, then independently verify at least one claim. Individual exit questions are completed without AI.

In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in the repository root or in notebooks/.
root = Path.cwd() if (Path.cwd() / 'la_labs.py').exists() else Path.cwd().parent
if not (root / 'la_labs.py').exists():
    raise RuntimeError('Start Jupyter in the repository root or notebooks directory.')
if str(root) not in sys.path: sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import la_labs as la


## Predict first

Write the route from new coordinates to physical coordinates, through A, then back to new coordinates. Predict which order of S and S⁻¹ belongs in the formula. Test on a basis that is not orthogonal.

## Worked route: follow one vector through the coordinate changes

Continue the column interpretation in the [original transformation notebook](../6_linear_transformations.ipynb). Let $S=\begin{pmatrix}1&1\\0&1\end{pmatrix}$. Its columns are the new basis vectors $(1,0)^T$ and $(1,1)^T$. A coordinate vector $c=(c_1,c_2)^T$ therefore describes the physical vector $Sc=(c_1+c_2,c_2)^T$.

For $A=\operatorname{diag}(2,1)$, the route is: turn c into Sc, apply A, then express the answer in the new basis. Hence

$$C=S^{-1}AS=\begin{pmatrix}2&1\\0&1\end{pmatrix}.$$

Take $c=e_2$. Its physical vector is $(1,1)^T$; applying A gives $(2,1)^T$. This is one copy of each new basis vector, so its new coordinates are $(1,1)^T$. That matches $Ce_2$.

The equation $AS=SC$ says that these two routes agree for every c. This is how to reconstruct the formula if you forget it. The wrong order $SAS^{-1}$ gives an upper-right entry of −1 here and fails this test. A symmetric example might conceal that error, which is why this basis was chosen.

In [ ]:
A = np.diag([2.,1.]); S = np.array([[1.,1.],[0.,1.]])
C = la.change_basis(A,S)
wrong = S@A@np.linalg.inv(S)  # Deliberately wrong order for comparison.
c = np.array([0.,1.])
print('correct matrix:\n',C,'\nwrong-order matrix:\n',wrong)
print('physical output:',A@S@c,'correct reconstruction:',S@C@c)
assert np.allclose(A@S,S@C)
assert not np.allclose(A@S,S@wrong)

## Affine composition

Predict where the origin goes in each order. Change the translation and rotation, including rotation 0°. When do the orders happen to agree? The homogeneous matrices act on points with last coordinate 1 and displacements with last coordinate 0.

### Why adding one coordinate handles translation

A translation changes the origin, so it cannot be represented by a linear map on the original coordinates. We instead write a point as $(x,y,1)^T$. Multiplying by

$$H=\begin{pmatrix}0&-1&2\\1&0&1\\0&0&1\end{pmatrix}$$

gives $(-y+2,x+1,1)^T$: a quarter-turn followed by translation by $(2,1)$. The final coordinate supplies the constant translation term. A displacement has last coordinate zero, so the same matrix sends $(u,v,0)^T$ to $(-v,u,0)^T$ without translating it.

Compare orders on the origin. Rotating it first and translating gives $(2,1)$. Translating first and rotating gives $R(2,1)=(-1,2)$. The pictures differ because the translation vector itself is rotated in the second order. This is the same composition-order issue that appeared with matrix products, now with points and directions distinguished.

In [ ]:
def explore(angle=90, tx=2, ty=1):
    fig, H = la.affine_figure(angle,tx,ty)
    print('Homogeneous matrix, rotation then translation:\n',H.round(4))
    print('origin ->',H@np.array([0.,0.,1.]))
    print('direction e1 ->',H@np.array([1.,0.,0.]))
    display(fig); plt.close(fig)
sliders = la.interactive_plot(explore, {'angle': (-180,180,15,90), 'tx': (-3,3,.5,2), 'ty': (-3,3,.5,1)})

## Optional after projection: least squares

These three data points do not lie on one line. Find a best-fitting line and verify residual orthogonality. This is an optional application, not a new mandatory topic. Do not confuse the residual of a fit with a measurement uncertainty estimate.

### Optional worked connection: a residual perpendicular to a plane

Write the fitted values as $X\beta$ with columns $\mathbf1$ and $(0,1,2)^T$. We can reach only their two-dimensional span in $\mathbb R^3$; the data vector y need not lie there. The projection idea now asks for the closest point in that span.

The residual must be perpendicular to both columns, giving $X^T(y-X\beta)=0$. In this example the resulting equations are $3\beta_0+3\beta_1=5$ and $3\beta_0+5\beta_1=6$. Subtraction gives $\beta_1=1/2$ and then $\beta_0=7/6$.

The residual is $(-1/6,1/3,-1/6)^T$, whose dot product with each column is zero. The computation below checks the same fit using a numerical least-squares routine. This derivation explains the geometry; explicitly inverting the normal-equation matrix is not the recommended general numerical algorithm.

In [ ]:
X = np.array([[1.,0.],[1.,1.],[1.,2.]])
y = np.array([1.,2.,2.])
beta, _, rank, singular_values = np.linalg.lstsq(X,y,rcond=None)
residual = y-X@beta
print('intercept, slope:',beta,'rank:',rank)
print('residual:',residual,'X.T @ residual:',X.T@residual)
assert np.allclose(X.T@residual,0)
assert np.allclose(beta,[7/6,1/2])

## Explain and transfer

Prove that affine maps preserve combinations whose weights sum to 1. Explain why nonnegative weights are a separate restriction.

**Individual exit:** apply a translation to a point and to a displacement and explain the difference without software.